<a href="https://colab.research.google.com/github/mohamedalaaaz/testpytroch/blob/main/Al%20engine%20game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

# model: trained PyTorch model
dummy_input = torch.randn(1, 3, 224, 224)  # adjust input shape
torch.onnx.export(model, dummy_input, "model.onnx")

In [ ]:

using Unity.Barracuda;

public class AIController : MonoBehaviour {
    public NNModel modelAsset;
    private IWorker worker;

    void Start() {
        var model = ModelLoader.Load(modelAsset);
        worker = WorkerFactory.CreateWorker(WorkerFactory.Type.ComputePrecompiled, model);
    }

    void Update() {
        // Example input: player position
        Tensor input = new Tensor(1, 2, new float[] { player.x, player.y });
        worker.Execute(input);
        Tensor output = worker.PeekOutput();

        // Use output to move NPC
        Vector2 action = new Vector2(output[0], output[1]);
        transform.position += new Vector3(action.x, 0, action.y);

        input.Dispose();
        output.Dispose();
    }
}

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

# Define model
class NPCModel(nn.Module):
    def __init__(self):
        super(NPCModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

# Train dummy model
model = NPCModel()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Fake training data: player pos -> NPC moves opposite direction
player_positions = torch.randn(1000, 2)
npc_targets = -player_positions

for epoch in range(100):
    optimizer.zero_grad()
    output = model(player_positions)
    loss = criterion(output, npc_targets)
    loss.backward()
    optimizer.step()

print("Training complete!")

In [ ]:
dummy_input = torch.randn(1, 2)  # input shape = [batch, features]
torch.onnx.export(
    model, dummy_input, "npc_model.onnx",
    input_names=["player_pos"],
    output_names=["npc_move"],
    dynamic_axes={"player_pos": {0: "batch"}, "npc_move": {0: "batch"}}
)
print("Model exported to npc_model.onnx")

In [ ]:
using Unity.Barracuda;
using UnityEngine;

public class NPC_AI : MonoBehaviour {
    public NNModel modelAsset;
    private IWorker worker;
    public Transform player; // drag player object here

    void Start() {
        var model = ModelLoader.Load(modelAsset);
        worker = WorkerFactory.CreateWorker(WorkerFactory.Type.Auto, model);
    }

    void Update() {
        // Get player position
        float[] inputArray = { player.position.x, player.position.z };
        Tensor input = new Tensor(1, 2, inputArray);

        // Run model
        worker.Execute(input);
        Tensor output = worker.PeekOutput("npc_move");

        // NPC movement
        Vector2 moveDir = new Vector2(output[0], output[1]);
        transform.position += new Vector3(moveDir.x, 0, moveDir.y) * Time.deltaTime;

        input.Dispose();
        output.Dispose();
    }

    void OnDestroy() {
        worker.Dispose();
    }
}